In [ ]:
from abc import ABC, abstractmethod


class ElevatorState(ABC):
    def __init__(self, elevator):
        self.elevator = elevator

    @abstractmethod
    def open_door(self):
        pass

    @abstractmethod
    def close_door(self):
        pass

    @abstractmethod
    def move(self):
        pass

    @abstractmethod
    def stop(self):
        pass

    @abstractmethod
    def get_floor(self):
        pass

    @abstractmethod
    def set_floor(self, floor):
        pass


class ElevatorIdleState(ElevatorState):
    def __init__(self, elevator):
        super().__init__(elevator)

    def open_door(self):
        pass

    def close_door(self):
        pass

    def move(self):
        pass

    def stop(self):
        pass

    def get_floor(self):
        pass

    def set_floor(self, floor):
        pass


class ElevatorMoveUpState(ElevatorState):
    def __init__(self, elevator):
        super().__init__(elevator)

    def open_door(self):
        pass

    def close_door(self):
        pass

    def move(self):
        # move up logic
        pass

    def stop(self):
        pass

    def get_floor(self):
        pass

    def set_floor(self, floor):
        pass


class ElevatorMoveDownState(ElevatorState):
    def __init__(self, elevator):
        super().__init__(elevator)

    def open_door(self):
        pass

    def close_door(self):
        pass

    def move(self):
        # move down logic
        pass

    def stop(self):
        pass

    def get_floor(self):
        pass

    def set_floor(self, floor):
        pass

class ElevatorDoorOpenState(ElevatorState):
    def __init__(self, elevator):
        super().__init__(elevator)

    def open_door(self):
        pass

    def close_door(self):
        pass

    def move(self):
        pass

    def stop(self):
        pass

    def get_floor(self):
        pass

    def set_floor(self, floor):
        pass



class Elevator:
    def __init__(self):
        self.current_floor = 0
        self.current_state = ElevatorIdleState(self)


class ElevatorSystem:
    def __init__(self):
        self.elevators = []
        self.requests = []


## Notes to Self
I realized that I didn't add the transition function names only the states during my verbal session.


### 3. State Machine:

1. Elevator states:
   - Idle (doors closed, no active movement)
   - MovingUp (doors closed)
   - MovingDown (doors closed)
   - DoorOpen (servicing pickup or dropoff at current floor)
   These are the physical elevator states. Boarding and alighting are not separate states because passenger simulation is out of scope.

2. Legal transitions:
   - Idle → MovingUp (request added to queue and processed, with starting floor above current floor), process_new_request
   - Idle → MovingDown (request added to queue and processed, with starting floor below current floor), process_new_request
   - Idle → DoorOpen (request added to queue and processed, with starting floor equal to current floor), process_new_request
   - MovingUp → DoorOpen (reached request start floor or end floor of currently serving request), resolve_floor
   - MovingDown → DoorOpen (reached request start floor or end floor of currently serving request), resolve_floor
   - DoorOpen → Idle (service complete and no next request to continue with immediately), service_complete
   - DoorOpen → MovingUp (next request in queue requires moving up), process_next_request
   - DoorOpen → MovingDown (next request in queue requires moving down), process_next_request

3. Illegal transitions:
   - MovingUp → MovingDown
   - MovingDown → MovingUp
   - MovingUp → Idle
   - MovingDown → Idle
   - Idle → Idle
   - DoorOpen → DoorOpen
   Any transition that implies the elevator is moving while doors are open is illegal.

## However, one has to process the intermediate floors when moving up or down
## Notes to Self
I realized that I didn't add the transition function names only the states during my verbal session.


### 3. State Machine:

1. Elevator states:
   - Idle (doors closed, no active movement)
   - MovingUp (doors closed)
   - MovingDown (doors closed)
   - DoorOpen (servicing pickup or dropoff at current floor)
   These are the physical elevator states. Boarding and alighting are not separate states because passenger simulation is out of scope.

2. Legal transitions:
   - Idle → MovingUp (request added to queue and processed, with starting floor above current floor), process_new_request
   - Idle → MovingDown (request added to queue and processed, with starting floor below current floor), process_new_request
   - Idle → DoorOpen (request added to queue and processed, with starting floor equal to current floor), process_new_request
   - MovingUp → MovingUp (process intermediate floors when moving up), move
   - MovingDown → MovingDown (process intermediate floors when moving down), move
   - Idle → Idle (no requests to process), no_op
   - MovingUp → DoorOpen (reached request start floor or end floor of currently serving request), resolve_floor
   - MovingDown → DoorOpen (reached request start floor or end floor of currently serving request), resolve_floor
   - DoorOpen → Idle (service complete and no next request to continue with immediately), service_complete
   - DoorOpen → MovingUp (next request in queue requires moving up), process_next_request
   - DoorOpen → MovingDown (next request in queue requires moving down), process_next_request

3. Illegal transitions:
   - MovingUp → MovingDown
   - MovingDown → MovingUp
   - MovingUp → Idle
   - MovingDown → Idle
   - Idle → Idle
   - DoorOpen → DoorOpen
   Any transition that implies the elevator is moving while doors are open is illegal.



## Additional note
1. for overengineering movingup and moving down state, we simplify to closed for the moment.

In [ ]:
from abc import ABC, abstractmethod
from collections import deque
from enum import Enum


class Request:
    def __init__(self, start_floor, end_floor):
        self.start_floor = start_floor
        self.end_floor = end_floor

class Direction(Enum):
    IDLE = 0
    UP = 1
    DOWN = 2

# Physical State of elevator
class ElevatorState(ABC):
    def __init__(self, elevator, floor=0, direction=Direction.IDLE):
        self.elevator = elevator
        self.floor = floor
        self.direction = direction
        
    
    # Assumption that processing takes a single time tick
    @abstractmethod
    def process(self, request):
        pass

    @abstractmethod
    def move(self):
        pass
        

    @abstractmethod
    def onboard(self):
        pass

    @abstractmethod
    def alight(self):
        pass


class ElevatorIdleState(ElevatorState):
    def __init__(self, elevator, floor=0):
        super().__init__(elevator, floor, Direction.IDLE)

    def process(self, request):
        # process new request
        self.elevator.target_floor = request.end_floor
        if self.elevator.target_floor > self.floor:
            self.elevator.current_state = ElevatorMoveUpState(self.elevator)
        else:
            self.elevator.current_state = ElevatorMoveDownState(self.elevator)
    
    def move(self):
        # move to next floor
        raise Exception("Currently Moving: Invalid operation in ElevatorIdleState")
    
    def onboard(self):
        # onboard logic
        raise Exception("Currently Onboarding: Invalid operation in ElevatorIdleState")
    
    def alight(self):
        # alight logic
        raise Exception("Currently Alighting: Invalid operation in ElevatorIdleState")

# Move up and move down are doorCloseStates subsets of partition, closed only when moving
class ElevatorDoorClosedState(ElevatorState):
    def __init__(self, elevator, floor=0, direction=Direction.IDLE):
        super().__init__(elevator, floor, direction)

    def process(self, request):
        # process new request
        raise Exception("Currently Processing Request: Invalid operation in ElevatorMoveUpState")

    def move(self):
        # move up logic
        if self.direction == Direction.UP:
            self.floor += 1
            if self.floor == self.elevator.target_floor:
                self.elevator.current_state = ElevatorDoorOpenState(self.elevator, self.floor, Direction.IDLE)
        elif self.direction == Direction.DOWN:
            self.floor -= 1
            if self.floor == self.elevator.target_floor:
                self.elevator.current_state = ElevatorDoorOpenState(self.elevator, self.floor, Direction.IDLE)
       
    def onboard(self):
        # onboard logic
        raise Exception("Currently Onboarding: Invalid operation in ElevatorDoorClosedState")
    
    def alight(self):
        # alight logic
        raise Exception("Currently Alighting: Invalid operation in ElevatorDoorClosedState")



class ElevatorDoorOpenState(ElevatorState):
    def __init__(self, elevator, floor=0):
        super().__init__(elevator, floor, Direction.IDLE)

    def process(self, request):
        self.elevator.target_floor = request.end_floor
        if self.elevator.target_floor > self.floor:
            self.elevator.current_state = ElevatorDoorClosedState(self.elevator, self.floor, Direction.UP)
        else:
            self.elevator.current_state = ElevatorDoorClosedState(self.elevator, self.floor, Direction.DOWN)

    def move(self):
        # move to next floor
        raise Exception("Currently Moving: Invalid operation in ElevatorDoorOpenState")

    def onboard(self):
        # onboard logic
        raise Exception("Currently Onboarding: Invalid operation in ElevatorDoorOpenState")
    
    def alight(self):
        # alight logic
        raise Exception("Currently Alighting: Invalid operation in ElevatorDoorOpenState")

# inverted control pattern, elevator delegates to state
# Span:  ElevatorContext --> ElevatorState. Before would be ElevatorState --> ElevatorContext (Naive implementation). Runtime --> Code instead of Code --> Runtime.

class ElevatorContext(ElevatorState):
    def __init__(self, elevator, floor=0, direction=Direction.IDLE):
        super().__init__(elevator, floor, direction)
    
    def process(self):
        self.elevator.current_state.process()
    
    def move(self):
        self.elevator.current_state.move()
    
    def onboard(self):
        self.elevator.current_state.onboard()
    
    def alight(self):
        self.elevator.current_state.alight()

class Elevator:
    def __init__(self):
        self.current_state = ElevatorIdleState(self)
        self.requests_queue = deque()

class ElevatorSystem:
    def __init__(self):
        self.elevators = []
        self.requests = []
        self.time_tick = 0
